In [33]:
import getpass
import pandas as pd
from google import genai


moviesFil = pd.read_csv('Datas/movies.csv')
linksFil = pd.read_csv('Datas/links.csv')
ratingsFil = pd.read_csv('Datas/ratings.csv')
tagsFil = pd.read_csv('Datas/tags.csv')
GEMINI_API_KEY = getpass.getpass("Introduce tu GEMINI_API_KEY: ")
TMDB_READ_ACCESS_TOKEN = getpass.getpass(
    "Introduce tu TMDB Read Access Token: "
)


In [34]:
import requests
client = genai.Client(api_key=GEMINI_API_KEY)


In [42]:
# Función para resumir en español
def summarize_overview_es(overview, title=""):

    if not overview:
        return ""

    prompt = f"""
Resume en español la siguiente sinopsis de una película en un máximo de 2 frases.
No añadas información que no aparezca en la sinopsis.

Título: {title}

Sinopsis:
{overview}
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    return response.text.strip()


In [43]:
#Unimos moviesFil con linksFil mediante movieId:
movies_links = moviesFil.merge(
    linksFil,
    on="movieId",
    how="inner"
)

movies10 = movies_links[
    movies_links["tmdbId"].notna()
].head(10).copy()

In [44]:
def fetch_movie_details(tmdb_id):

    url = f"https://api.themoviedb.org/3/movie/{int(tmdb_id)}"

    headers = {
        "Authorization": f"Bearer {TMDB_READ_ACCESS_TOKEN}"
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()

        return {
            "overview": data.get("overview") or "",
            "homepage": data.get("homepage") or ""
        }

    return {
        "overview": "",
        "homepage": ""

    }

movies10["overview"] = ""
movies10["homepage"] = ""


for index, row in movies10.iterrows():

    detalles = fetch_movie_details(row["tmdbId"])

    movies10.loc[index, "overview"] = detalles["overview"]
    movies10.loc[index, "homepage"] = detalles["homepage"]

print(movies10[["title", "overview", "homepage"]])



                                title  \
0                    Toy Story (1995)   
1                      Jumanji (1995)   
2             Grumpier Old Men (1995)   
3            Waiting to Exhale (1995)   
4  Father of the Bride Part II (1995)   
5                         Heat (1995)   
6                      Sabrina (1995)   
7                 Tom and Huck (1995)   
8                 Sudden Death (1995)   
9                    GoldenEye (1995)   

                                            overview  \
0  Led by Woody, Andy's toys live happily in his ...   
1  When siblings Judy and Peter discover an encha...   
2  A family wedding reignites the ancient feud be...   
3  Cheated on, mistreated and stepped on, the wom...   
4  Just when George Banks has recovered from his ...   
5  Obsessive master thief Neil McCauley leads a t...   
6  After her return from school in Paris, a playb...   
7  A mischievous young boy, Tom Sawyer, witnesses...   
8  When a man's daughter is suddenly taken d

In [45]:
movies10["overview_es"] = ""

for index, row in movies10.iterrows():

    movies10.loc[index, "overview_es"] = summarize_overview_es(
        row["overview"],
        row["title"]
    )

    muestra = movies10[
    ["title", "overview", "overview_es"]
].head(3).copy()

muestra["overview"] = muestra["overview"].str[:200]

print(muestra.to_string(index=False))


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
tmdb_id = int(movies10.iloc[0]["tmdbId"])

url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"

headers = {
    "Authorization": f"Bearer {TMDB_READ_ACCESS_TOKEN}"
}

response = requests.get(url, headers=headers)

print("Código:", response.status_code)
print(response.text[:500])


Código: 200
{"adult":false,"backdrop_path":"/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg","belongs_to_collection":{"id":10194,"name":"Toy Story Collection","poster_path":"/rki5qLuwb0xnnE9seehxO9TlLhW.jpg","backdrop_path":"/hApclyB9NEZEQujAVajzi5iWE4a.jpg"},"budget":30000000,"genres":[{"id":10751,"name":"Family"},{"id":35,"name":"Comedy"},{"id":16,"name":"Animation"},{"id":12,"name":"Adventure"}],"homepage":"http://toystory.disney.com/toy-story","id":862,"imdb_id":"tt0114709","origin_country":["US"],"original_language":
